# 03 - Fine-tune CATT (Character-based Arabic Tashkeel Transformer)

**Base Model:** `abjadai/catt` (already fine-tuned Character Transformer)
**Architecture:** Character-level Encoder-Decoder Transformer
**Current Score:** 0.41 WER (Dev)

**Techniques:**
- Further fine-tuning on KSSA dataset
- Curriculum Learning (short to long texts)
- Label Smoothing
- Mixed Precision (FP16)
- Gradient Checkpointing

In [1]:
!pip install -q transformers torch accelerate tqdm catt-tashkeel

In [2]:
# Setup
import os, sys, json, re, torch, gc, subprocess, zipfile
from datetime import datetime
from tqdm import tqdm
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

IS_RUNPOD = False  # Cloud detection removed
PROJECT_DIR = Path('.')
sys.path.insert(0, str(PROJECT_DIR))

DATA_DIR = PROJECT_DIR / 'Public_Data_TrainDev'
TRAIN_DIR = DATA_DIR / 'Train'
DEV_INPUT = DATA_DIR / 'dev input-output' / 'Dev_input.json'
DEV_OUTPUT = DATA_DIR / 'dev input-output' / 'Dev_output.json'
TEST_DIR = PROJECT_DIR / 'Test'
TEST_INPUT = TEST_DIR / 'test_input.json'
OUTPUT_DIR = PROJECT_DIR / 'outputs'
SUBMISSION_DIR = PROJECT_DIR / 'submissions'
CHECKPOINTS_DIR = PROJECT_DIR / 'checkpoints' / 'catt_ft'
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
SUBMISSION_DIR.mkdir(exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Environment: 'Local' | Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB)")

def clear_gpu():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()
        print(f"GPU Memory: {torch.cuda.memory_allocated()/1024**3:.2f} GB allocated")

Environment: Local | Device: cuda
GPU: NVIDIA RTX A5000 (23.5 GB)


In [3]:
# Load Training Data
import pandas as pd

train_tsv = TRAIN_DIR / 'train_list.tsv'
train_df = pd.read_csv(train_tsv, sep='\t')
print(f"Training samples in TSV: {len(train_df)}")

def load_training_data():
    data = []
    text_dir = TRAIN_DIR / 'train_text'
    
    for idx, row in tqdm(train_df.iterrows(), total=len(train_df), desc="Loading data"):
        txt_file = text_dir / f"{row['stem']}.txt"
        if txt_file.exists():
            with open(txt_file, 'r', encoding='utf-8') as f:
                diacritized = f.read().strip()
            # Remove diacritics to get undiacritized version
            undiacritized = re.sub(r'[\u064B-\u0652]', '', diacritized)
            data.append({
                'id': row['stem'],
                'text_undiacritized': undiacritized,
                'text_diacritized': diacritized
            })
    return data

train_data = load_training_data()
# Curriculum learning: sort by length (easy to hard)
train_data = sorted(train_data, key=lambda x: len(x['text_undiacritized']))
print(f"Loaded {len(train_data)} training samples")

Training samples in TSV: 2327


Loading data: 100%|██████████| 2327/2327 [00:16<00:00, 143.80it/s]

Loaded 2327 training samples


In [4]:
# Load Fine-Tashkeel (HuggingFace) - CATT is inference-only, use Fine-Tashkeel for training
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)
from datasets import Dataset

MODEL_NAME = 'basharalrfooh/Fine-Tashkeel'  # HuggingFace ByT5, already fine-tuned for tashkeel
MODEL_KEY = 'catt_ft'

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
).to(device)

model.gradient_checkpointing_enable()
print(f"Model loaded: {sum(p.numel() for p in model.parameters()):,} parameters")

Loading basharalrfooh/Fine-Tashkeel...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/308 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Model loaded: 720,870,144 parameters


In [5]:
# Prepare Dataset
MAX_LENGTH = 512

def preprocess_function(examples):
    model_inputs = tokenizer(
        examples['text_undiacritized'], 
        max_length=MAX_LENGTH, 
        truncation=True, 
        padding='max_length'
    )
    labels = tokenizer(
        examples['text_diacritized'], 
        max_length=MAX_LENGTH, 
        truncation=True, 
        padding='max_length'
    )
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

train_dataset = Dataset.from_list(train_data)
train_dataset = train_dataset.map(
    preprocess_function, 
    batched=True, 
    remove_columns=['id', 'text_undiacritized', 'text_diacritized']
)

# Load dev data for validation
with open(DEV_INPUT, 'r', encoding='utf-8') as f: dev_input = json.load(f)
with open(DEV_OUTPUT, 'r', encoding='utf-8') as f: dev_output = json.load(f)
dev_lookup = {item['id']: item['text_diacritized'] for item in dev_output}
dev_data = [{'text_undiacritized': item['text_undiacritized'], 
             'text_diacritized': dev_lookup.get(item['id'], '')} 
            for item in dev_input if item['id'] in dev_lookup]

eval_dataset = Dataset.from_list(dev_data)
eval_dataset = eval_dataset.map(
    preprocess_function, 
    batched=True, 
    remove_columns=['text_undiacritized', 'text_diacritized']
)

print(f"Train: {len(train_dataset)}, Eval: {len(eval_dataset)}")

Map:   0%|          | 0/2327 [00:00<?, ? examples/s]

Map:   0%|          | 0/260 [00:00<?, ? examples/s]

Train: 2327, Eval: 260


In [6]:
# Training Arguments
training_args = Seq2SeqTrainingArguments(
    output_dir=str(CHECKPOINTS_DIR),
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,  # Effective batch: 32
    learning_rate=2e-5,  # Lower LR for fine-tuning
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=True if device == "cuda" else False,
    label_smoothing_factor=0.1,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=3,
    load_best_model_at_end=True,
    logging_steps=50,
    report_to="none",
    predict_with_generate=True,
    generation_max_length=MAX_LENGTH,
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [7]:
# Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

# Train with checkpoint resume
clear_gpu()
checkpoint = None
checkpoints = list(CHECKPOINTS_DIR.glob('checkpoint-*'))
if checkpoints:
    checkpoint = str(max(checkpoints, key=lambda x: int(x.name.split('-')[1])))
    print(f"Resuming from: {checkpoint}")

trainer.train(resume_from_checkpoint=checkpoint)

GPU Memory: 2.75 GB allocated


Step,Training Loss,Validation Loss


OutOfMemoryError: CUDA out of memory. Tried to allocate 24.00 MiB. GPU 0 has a total capacity of 23.55 GiB of which 8.12 MiB is free. Including non-PyTorch memory, this process has 13.36 GiB memory in use. Process 10507 has 10.17 GiB memory in use. Of the allocated memory 12.54 GiB is allocated by PyTorch, and 574.57 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# Save
final_path = CHECKPOINTS_DIR / 'final'
trainer.save_model(str(final_path))
tokenizer.save_pretrained(str(final_path))
print(f"Saved to: {final_path}")
clear_gpu()

In [9]:
clear_gpu()

GPU Memory: 10.91 GB allocated


## Inference on Dev and Test Sets

In [ ]:
# Load fine-tuned model for inference
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

final_model_path = CHECKPOINTS_DIR / 'final'
if final_model_path.exists():
    print(f"Loading fine-tuned model from {final_model_path}")
    model = AutoModelForSeq2SeqLM.from_pretrained(
        final_model_path,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32
    ).to(device)
    tokenizer = AutoTokenizer.from_pretrained(final_model_path)
else:
    print("Using model from training (already in memory)")

model.eval()

In [ ]:
@torch.inference_mode()
def diacritize(text):
    inputs = tokenizer(text, return_tensors="pt", max_length=MAX_LENGTH, truncation=True).to(device)
    outputs = model.generate(**inputs, max_length=MAX_LENGTH, num_beams=4)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
# Dev Set
CHECKPOINT_FILE = OUTPUT_DIR / f"{MODEL_KEY}_checkpoint.json"

def load_checkpoint():
    if CHECKPOINT_FILE.exists():
        with open(CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
            data = json.load(f)
            return set(data.get('processed_ids', [])), data.get('predictions', [])
    return set(), []

def save_checkpoint(predictions):
    with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump({'processed_ids': [p['id'] for p in predictions], 'predictions': predictions}, f, ensure_ascii=False)

processed_ids, dev_predictions = load_checkpoint()
print(f"Dev: {len(dev_input)} samples, {len(processed_ids)} already done")

for item in tqdm(dev_input, desc="Dev Set"):
    if item['id'] in processed_ids: continue
    try:
        result = diacritize(item['text_undiacritized'])
        dev_predictions.append({'id': item['id'], 'text_diacritized': result})
        save_checkpoint(dev_predictions)
    except Exception as e:
        dev_predictions.append({'id': item['id'], 'text_diacritized': item['text_undiacritized']})
        save_checkpoint(dev_predictions)

with open(OUTPUT_DIR / f"{MODEL_KEY}_dev_predictions.json", 'w', encoding='utf-8') as f:
    json.dump(dev_predictions, f, ensure_ascii=False, indent=2)

In [ ]:
# Compute Dev Metrics
DIACRITIC_PATTERN = re.compile(r'[\u064B-\u0652]')
ARABIC_LETTER_PATTERN = re.compile(r'[\u0621-\u064A]')

def extract_diacritics(text):
    result = []
    i = 0
    while i < len(text):
        if ARABIC_LETTER_PATTERN.match(text[i]):
            diacritics = []
            j = i + 1
            while j < len(text) and DIACRITIC_PATTERN.match(text[j]):
                diacritics.append(text[j])
                j += 1
            result.append((text[i], ''.join(diacritics)))
            i = j
        else:
            i += 1
    return result

def compute_metrics(predictions, ground_truth):
    gt_lookup = {item['id']: item['text_diacritized'] for item in ground_truth}
    total_chars, total_words, char_errors, word_errors, n = 0, 0, 0, 0, 0
    for pred in predictions:
        if pred['id'] not in gt_lookup: continue
        pred_pairs = extract_diacritics(pred['text_diacritized'])
        ref_pairs = extract_diacritics(gt_lookup[pred['id']])
        for i in range(min(len(pred_pairs), len(ref_pairs))):
            if pred_pairs[i][1] != ref_pairs[i][1]: char_errors += 1
        char_errors += abs(len(pred_pairs) - len(ref_pairs))
        total_chars += max(len(pred_pairs), len(ref_pairs))
        pred_words, ref_words = pred['text_diacritized'].split(), gt_lookup[pred['id']].split()
        for i in range(min(len(pred_words), len(ref_words))):
            if pred_words[i] != ref_words[i]: word_errors += 1
        word_errors += abs(len(pred_words) - len(ref_words))
        total_words += max(len(pred_words), len(ref_words))
        n += 1
    return {'DER': char_errors/total_chars if total_chars else 0, 'WER': word_errors/total_words if total_words else 0}

metrics = compute_metrics(dev_predictions, dev_output)
print(f"\nDEV RESULTS: DER={metrics['DER']*100:.2f}% | WER={metrics['WER']*100:.2f}%")

In [ ]:
# Test Set
with open(TEST_INPUT, 'r', encoding='utf-8') as f: test_input = json.load(f)
TEST_CHECKPOINT_FILE = OUTPUT_DIR / f"{MODEL_KEY}_test_checkpoint.json"

def load_test_checkpoint():
    if TEST_CHECKPOINT_FILE.exists():
        with open(TEST_CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
            data = json.load(f)
            return set(data.get('processed_ids', [])), data.get('predictions', [])
    return set(), []

def save_test_checkpoint(predictions):
    with open(TEST_CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump({'processed_ids': [p['id'] for p in predictions], 'predictions': predictions}, f, ensure_ascii=False)

test_processed_ids, test_predictions = load_test_checkpoint()
print(f"Test: {len(test_input)} samples, {len(test_processed_ids)} already done")

for item in tqdm(test_input, desc="Test Set"):
    if item['id'] in test_processed_ids: continue
    try:
        result = diacritize(item['text_undiacritized'])
        test_predictions.append({'id': item['id'], 'text_diacritized': result})
        save_test_checkpoint(test_predictions)
    except:
        test_predictions.append({'id': item['id'], 'text_diacritized': item['text_undiacritized']})
        save_test_checkpoint(test_predictions)

# Submission
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
json_file = SUBMISSION_DIR / f"{MODEL_KEY}_submission.json"
with open(json_file, 'w', encoding='utf-8') as f:
    json.dump(test_predictions, f, ensure_ascii=False, indent=2)
zip_file = SUBMISSION_DIR / f"{MODEL_KEY}_submission_{timestamp}.zip"
with zipfile.ZipFile(zip_file, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(json_file, 'submission.json')
print(f"SUBMISSION: {zip_file}")

In [ ]:
# Google Drive sync removed for public release
